# Flight Delay Prediction — Modeling Notebook

**Temporal split:** Train 2018–2022 · Validate 2023 · Test 2024  
**Prediction window:** T–2 hours relative to scheduled departure  
**Target:** `ArrDel15` — binary (1 = arrival ≥ 15 min late)

---
## 1. Connect to GitHub

In [1]:
!git clone https://github.com/javageek2018/AirlineArrivalDelay.git

Cloning into 'AirlineArrivalDelay'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 50 (delta 17), reused 45 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 4.51 MiB | 10.28 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
%cd AirlineArrivalDelay

/content/AirlineArrivalDelay


In [3]:
!git fetch --all
!git branch -a

Fetching origin
* main
  remotes/origin/EDA
  remotes/origin/HEAD -> origin/main
  remotes/origin/main
  remotes/origin/“flight_data”


---
## 2. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 3. Set Up Spark

In [5]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /usr/bin/rmid

In [6]:
!pip install pyspark --quiet

In [9]:
import os
import subprocess
import pyspark

java_home = subprocess.run(
    ["dirname", "$(dirname $(readlink -f $(which java)))"],
    capture_output=True, text=True, shell=False
)

result = subprocess.run(
    "java -XshowSettings:property -version 2>&1 | grep 'java.home'",
    shell=True, capture_output=True, text=True
)
java_home_path = result.stdout.strip().split("=")[-1].strip()

os.environ["JAVA_HOME"] = java_home_path
print(f"JAVA_HOME set to : {java_home_path}")
print(f"PySpark version  : {pyspark.__version__}")

JAVA_HOME set to : /usr/lib/jvm/java-17-openjdk-amd64
PySpark version  : 4.0.2


In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FlightDelayModeling")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.sparkContext.setLogLevel("WARN")
spark

---
## 4. Read Training Data

Reads the pre-split training parquet from Google Drive.  
Validation and test sets are loaded separately only when needed for evaluation.

In [19]:
BASE_PATH = "/content/drive/MyDrive/OMDS Capstone/Data/flights_split"

TRAIN_PATH    = f"{BASE_PATH}/train_2018_2022.parquet"
VALIDATE_PATH = f"{BASE_PATH}/validate_2023.parquet"
TEST_PATH     = f"{BASE_PATH}/test_2024.parquet"

# Load training data — validation and test intentionally deferred
train_df = spark.read.parquet(TRAIN_PATH)

print("Training data loaded.")

Training data loaded.


---
## 5. Basic Data Verification

Confirm schema, shape, sample rows, and target class balance before final feature engineering of rolling temporal features.
This is a Sanity Check before moving forward

In [20]:
# 5.1  Schema
print(f"{'Column':<40} {'Type'}")
print("-" * 55)
for field in train_df.schema.fields:
    print(f"{field.name:<40} {field.dataType.simpleString()}")

Column                                   Type
-------------------------------------------------------
Year                                     bigint
Quarter                                  bigint
Month                                    bigint
DayofMonth                               bigint
DayOfWeek                                bigint
FlightDate                               bigint
Reporting_Airline                        string
Flight_Number_Reporting_Airline          string
Origin                                   string
Dest                                     string
CRSDepTime                               bigint
DepTimeBlk                               string
CRSArrTime                               bigint
ArrDel15                                 bigint
CRSElapsedTime                           double
Distance                                 double
DistanceGroup                            bigint
date                                     string
dep_hour                          

In [21]:
# 5.2  Row count
total_rows = train_df.count()
total_cols = len(train_df.columns)
print(f"Rows    : {total_rows:>12,}")
print(f"Columns : {total_cols:>12,}")

Rows    :   31,149,502
Columns :           56


In [22]:
# 5.3  Sample rows
train_df.show(5, truncate=False, vertical=True)

-RECORD 0----------------------------------------------
 Year                            | 2018                
 Quarter                         | 1                   
 Month                           | 1                   
 DayofMonth                      | 1                   
 DayOfWeek                       | 1                   
 FlightDate                      | 1514764800000000000 
 Reporting_Airline               | WN                  
 Flight_Number_Reporting_Airline | 1491.0              
 Origin                          | ABQ                 
 Dest                            | BWI                 
 CRSDepTime                      | 730                 
 DepTimeBlk                      | 0700-0759           
 CRSArrTime                      | 1310                
 ArrDel15                        | 0                   
 CRSElapsedTime                  | 220.0               
 Distance                        | 1670.0              
 DistanceGroup                   | 7            

In [23]:
# 5.4  Year distribution — confirm temporal split is clean
from pyspark.sql import functions as F

(
    train_df
    .groupBy("Year")
    .agg(F.count("*").alias("flights"))
    .orderBy("Year")
    .show()
)

+----+-------+
|Year|flights|
+----+-------+
|2018|7071464|
|2019|7268232|
|2020|4399575|
|2021|5878219|
|2022|6532012|
+----+-------+



In [24]:
# 5.5  Target class balance (ArrDel15)
# Class imbalance is expected (~20% delayed); this will confirm ratio
# and inform class-weight settings for all models as necessary.

balance = (
    train_df
    .groupBy("ArrDel15")
    .agg(F.count("*").alias("count"))
    .orderBy("ArrDel15")
    .collect()
)

print(f"{'Label':<12} {'Count':>12} {'Pct':>8}")
print("-" * 35)
for row in balance:
    label = "Not Delayed" if row["ArrDel15"] == 0 else "Delayed"
    pct   = row["count"] / total_rows * 100
    print(f"{label:<12} {row['count']:>12,} {pct:>7.2f}%")

delayed_count     = next(r["count"] for r in balance if r["ArrDel15"] == 1)
not_delayed_count = next(r["count"] for r in balance if r["ArrDel15"] == 0)
imbalance_ratio   = not_delayed_count / delayed_count
print(f"\nImbalance ratio (neg:pos) : {imbalance_ratio:.2f}:1")
print(f"Suggested class_weight    : {{0: 1.0, 1: {imbalance_ratio:.2f}}}")

Label               Count      Pct
-----------------------------------
Not Delayed    25,589,033   82.15%
Delayed         5,560,469   17.85%

Imbalance ratio (neg:pos) : 4.60:1
Suggested class_weight    : {0: 1.0, 1: 4.60}


In [25]:
# 5.6  Target balance by year — check for drift in imbalance ratio
(
    train_df
    .groupBy("Year")
    .agg(
        F.count("*").alias("total"),
        F.sum("ArrDel15").alias("delayed"),
        (F.sum("ArrDel15") / F.count("*") * 100).alias("delay_pct")
    )
    .orderBy("Year")
    .show()
)

+----+-------+-------+------------------+
|Year|  total|delayed|         delay_pct|
+----+-------+-------+------------------+
|2018|7071464|1352165| 19.12142945223224|
|2019|7268232|1389253|19.114043140064872|
|2020|4399575| 431921|  9.81733462891302|
|2021|5878219|1010332| 17.18772301610403|
|2022|6532012|1376798|21.077701633126207|
+----+-------+-------+------------------+



In [26]:
# 5.7  Numeric summary statistics
# Focus on model-input numerics; skip raw BTS identifier columns
numeric_cols = [
    f.name for f in train_df.schema.fields
    if f.dataType.simpleString() in ("int", "bigint", "double", "float", "long")
]
train_df.select(numeric_cols).describe().show(truncate=False)

+-------+------------------+------------------+-----------------+------------------+------------------+----------------------+------------------+------------------+-------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-----------------+-----------------+------------------+-------------------+-------------------+--------------------+-------------------+---------------------+--------------------+---------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+-----------------+-------------------+-------------------+--------------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+----------------------+--------------------+
|summary|Year              |Quarter           |Mont

In [27]:
# 5.8  Null audit — single-pass across all columns
from pyspark.sql.types import FloatType, DoubleType, StringType

null_exprs = []
for field in train_df.schema.fields:
    c, dt = field.name, field.dataType
    if isinstance(dt, (FloatType, DoubleType)):
        cond = F.col(c).isNull() | F.isnan(F.col(c))
    elif isinstance(dt, StringType):
        cond = F.col(c).isNull() | (F.trim(F.col(c)) == "")
    else:
        cond = F.col(c).isNull()
    null_exprs.append(F.sum(cond.cast("long")).alias(c))

null_exprs.append(F.count(F.lit(1)).alias("__total__"))
agg = train_df.agg(*null_exprs).collect()[0].asDict()
n   = agg.pop("__total__")

issues = {k: v for k, v in agg.items() if v > 0}
if issues:
    print(f"{'Column':<40} {'Missing':>10} {'Pct':>8}")
    print("-" * 62)
    for col, cnt in sorted(issues.items(), key=lambda x: -x[1]):
        print(f"{col:<40} {cnt:>10,} {cnt/n*100:>7.2f}%")
else:
    print("✅ No missing values detected in training data.")

✅ No missing values detected in training data.


---
## 6. Rolling Feature Engineering

Two rolling features are computed here, both designed to be **temporally strict**:
lookback windows reference only data *prior* to the current flight's date/time,
so there is no leakage from future observations.

| Feature | Window | Partitioned by | Ordered by |
|---|---|---|---|
| `carrier_delay_rate_30d` | Preceding 30 days | `Reporting_Airline` | `FlightDate` |
| `carrier_delay_rate_90d` | Preceding 90 days | `Reporting_Airline` | `FlightDate` |
| `origin_departures_3h` | Preceding 3 hours | `Origin` | `CRSDepTime` (unix) |

> **Leakage note:** `rangeBetween(-N, -1)` excludes the current row's timestamp,
> ensuring the feature represents only what was knowable at T–2.

In [47]:
train_df.select("FlightDate").show(5, truncate=False)
print(train_df.schema["FlightDate"])

+-------------------+
|FlightDate         |
+-------------------+
|1514764800000000000|
|1514764800000000000|
|1514764800000000000|
|1514764800000000000|
|1514764800000000000|
+-------------------+
only showing top 5 rows
StructField('FlightDate', LongType(), True)


In [48]:
from pyspark.sql import Window
from pyspark.sql import functions as F

###
# 6.1  Shared timestamp column
#
# Convert FlightDate (unix timestamp in nanoseconds) to a unix timestamp in
# SECONDS. This is used as the numeric ordering axis for all date-range windows.
# Casting to long gives whole-day granularity, which is all we need for
# the carrier rolling window, which operated on calendar days.
###

train_df = train_df.withColumn(
    "flight_date_unix",
    (F.col("FlightDate") / 1_000_000_000).cast("long")
)

In [49]:
###
# 6.2  Carrier rolling delay rate  (30-day and 90-day trailing windows)
#
# Rationale (EDA Figs 31–32): Carriers show persistent and time-varying
# heterogeneity.  A short trailing rate captures current operational state
# (staffing, fleet, hub congestion); a longer window smooths noise.
#
# Window semantics:
#   - partitionBy("Reporting_Airline")  -> separate history per carrier
#   - orderBy("flight_date_unix")       -> chronological order
#   - rangeBetween(-N_days * 86400, -1) -> [-N days ago, 1 second before midnight]
#                                          strictly excludes today's flights
#
# NaN behaviour: flights in the first 30/90 days of 2018 will have fewer
# historical rows (or none) in the window.  avg() over an empty window returns
# NULL, which we fill with the global training-set delay rate as a neutral prior.
###

SECS_PER_DAY = 86_400

w_carrier_30d = (
    Window
    .partitionBy("Reporting_Airline")
    .orderBy("flight_date_unix")
    .rangeBetween(-30 * SECS_PER_DAY, -1)
)

w_carrier_90d = (
    Window
    .partitionBy("Reporting_Airline")
    .orderBy("flight_date_unix")
    .rangeBetween(-90 * SECS_PER_DAY, -1)
)

train_df = (
    train_df
    .withColumn("carrier_delay_rate_30d", F.avg("ArrDel15").over(w_carrier_30d))
    .withColumn("carrier_delay_rate_90d", F.avg("ArrDel15").over(w_carrier_90d))
)

# Fill cold-start NULLs (first 30/90 days of data) with the global delay rate
global_delay_rate = train_df.agg(F.avg("ArrDel15")).collect()[0][0]
print(f"Global training delay rate (cold-start fill): {global_delay_rate:.4f}")

train_df = train_df.fillna({
    "carrier_delay_rate_30d": global_delay_rate,
    "carrier_delay_rate_90d": global_delay_rate,
})

Global training delay rate (cold-start fill): 0.1785


In [50]:
###
# 6.3  Origin airport rolling congestion index  (preceding 3-hour window)
#
# Rationale (EDA Fig 25): Delays build near-linearly through the operating day
# as upstream disruptions propagate forward.  dep_hour captures the time of day
# but not how loaded the airport actually is at that moment.  A count of
# scheduled departures from the same origin in the preceding 3 hours proxies
# for gate / ramp / ATC congestion at T–2.
#
# Construction:
#   1. Combine FlightDate + CRSDepTime (integer HHMM) into a full unix timestamp.
#   2. Apply a range window of [-3 hours, -1 second] over that timestamp,
#      partitioned by Origin — strictly excludes the current departure.
#   3. count() over an empty window returns NULL → fill with 0.
#
# Note: CRSDepTime is an integer in HHMM format (e.g. 835 = 08:35, 1420 = 14:20).
# We convert to seconds-since-midnight: (HHMM // 100) * 3600 + (HHMM % 100) * 60
###

train_df = train_df.withColumn(
    "crs_dep_unix",
    F.col("flight_date_unix")
    + (F.col("CRSDepTime") / 100).cast("int") * 3600      # hours -> seconds
    + (F.col("CRSDepTime") % 100) * 60                    # minutes -> seconds
)

SECS_3H = 3 * 3600   # 10,800 seconds

w_congestion = (
    Window
    .partitionBy("Origin")
    .orderBy("crs_dep_unix")
    .rangeBetween(-SECS_3H, -1)
)

train_df = train_df.withColumn(
    "origin_departures_3h",
    F.count("*").over(w_congestion)
)

train_df = train_df.fillna({"origin_departures_3h": 0})

In [51]:
# 6.4  Drop construction columns

train_df = train_df.drop("flight_date_unix", "crs_dep_unix")

---
## 7. Verify Rolling Features

In [52]:
# 7.1  Null check on the three new features
new_features = ["carrier_delay_rate_30d", "carrier_delay_rate_90d", "origin_departures_3h"]

null_check = train_df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in new_features
]).collect()[0].asDict()

print("Null counts after fill:")
for col, n in null_check.items():
    status = "✅" if n == 0 else "⚠️ "
    print(f"  {status} {col:<30} {n}")

Null counts after fill:
  ✅ carrier_delay_rate_30d         0
  ✅ carrier_delay_rate_90d         0
  ✅ origin_departures_3h           0


In [53]:
# 7.2  Distribution summary of new features
train_df.select(new_features).describe().show()

+-------+----------------------+----------------------+--------------------+
|summary|carrier_delay_rate_30d|carrier_delay_rate_90d|origin_departures_3h|
+-------+----------------------+----------------------+--------------------+
|  count|              31149502|              31149502|            31149502|
|   mean|    0.1780150584386198|   0.17783364218394393|   55.44348445763274|
| stddev|   0.06707551965637182|   0.05983284121198324|  51.487036490982284|
|    min|  0.014705882352941176|  0.032808115691776385|                   0|
|    max|    0.5544063779357897|    0.5544063779357897|                 288|
+-------+----------------------+----------------------+--------------------+



In [54]:
# 7.3  Carrier rolling rate sanity check
#      The 30d rate should correlate with the 90d rate and both
#      should be in a plausible range (0.05–0.50)
print("Carrier delay rate range check:")
train_df.select(
    F.min("carrier_delay_rate_30d").alias("30d_min"),
    F.max("carrier_delay_rate_30d").alias("30d_max"),
    F.avg("carrier_delay_rate_30d").alias("30d_mean"),
    F.min("carrier_delay_rate_90d").alias("90d_min"),
    F.max("carrier_delay_rate_90d").alias("90d_max"),
    F.avg("carrier_delay_rate_90d").alias("90d_mean"),
).show()

Carrier delay rate range check:
+--------------------+------------------+-------------------+--------------------+------------------+-------------------+
|             30d_min|           30d_max|           30d_mean|             90d_min|           90d_max|           90d_mean|
+--------------------+------------------+-------------------+--------------------+------------------+-------------------+
|0.014705882352941176|0.5544063779357897|0.17801505843823937|0.032808115691776385|0.5544063779357897|0.17783364218310707|
+--------------------+------------------+-------------------+--------------------+------------------+-------------------+



In [55]:
# 7.4  Congestion index sanity check
#      Busy hub airports (ATL, ORD, DFW) in afternoon banks should
#      show materially higher counts than regional airports at 06:00
print("Congestion index: distribution by departure hour")
(
    train_df
    .groupBy("dep_hour")
    .agg(
        F.avg("origin_departures_3h").alias("avg_3h_deps"),
        F.max("origin_departures_3h").alias("max_3h_deps")
    )
    .orderBy("dep_hour")
    .show(24)
)

Congestion index: distribution by departure hour
+--------+------------------+-----------+
|dep_hour|       avg_3h_deps|max_3h_deps|
+--------+------------------+-----------+
|       0|   34.229871352369|        203|
|       1| 19.42851801071918|        140|
|       2| 3.762772785622593|         94|
|       3|3.6089045483664317|         12|
|       4| 3.907016060862215|         13|
|       5|2.1165582231918383|         31|
|       6| 9.192953594169856|         69|
|       7| 26.86981689336676|        135|
|       8| 52.08653708186353|        207|
|       9|  67.1345597224437|        231|
|      10| 70.97565803023022|        274|
|      11| 67.15002710048901|        288|
|      12| 63.00584734059792|        239|
|      13|58.157997367024514|        224|
|      14| 58.82977577777153|        214|
|      15| 59.60899012778779|        232|
|      16| 59.77071558528036|        253|
|      17| 54.34619339518417|        243|
|      18| 60.31754408940108|        236|
|      19| 65.3901580829542

In [56]:
# 7.5  Confirm rolling features show lift on delay rate
#      Flights in high-congestion periods should have a higher observed
#      delay rate than low-congestion periods
print("Delay rate by congestion quartile (origin_departures_3h):")

from pyspark.sql.functions import ntile

w_ntile = Window.orderBy("origin_departures_3h")
(
    train_df
    .withColumn("congestion_q", ntile(4).over(w_ntile))
    .groupBy("congestion_q")
    .agg(
        F.count("*").alias("flights"),
        F.avg("origin_departures_3h").alias("avg_departures"),
        (F.avg("ArrDel15") * 100).alias("delay_rate_pct")
    )
    .orderBy("congestion_q")
    .show()
)

Delay rate by congestion quartile (origin_departures_3h):
+------------+-------+------------------+------------------+
|congestion_q|flights|    avg_departures|    delay_rate_pct|
+------------+-------+------------------+------------------+
|           1|7787376| 4.899025679510017|14.654885034445492|
|           2|7787376| 26.06614063581879|16.994620524294703|
|           3|7787375|61.127767829339156|19.511003386892245|
|           4|7787375|129.68101394885954|20.243124287709275|
+------------+-------+------------------+------------------+



In [57]:
# 7.6  Final column inventory
print(f"Final training DataFrame: {train_df.count():,} rows × {len(train_df.columns)} columns\n")
print("All columns:")
for c in train_df.columns:
    print(f"  {c}")

Final training DataFrame: 31,149,502 rows × 59 columns

All columns:
  Year
  Quarter
  Month
  DayofMonth
  DayOfWeek
  FlightDate
  Reporting_Airline
  Flight_Number_Reporting_Airline
  Origin
  Dest
  CRSDepTime
  DepTimeBlk
  CRSArrTime
  ArrDel15
  CRSElapsedTime
  Distance
  DistanceGroup
  date
  dep_hour
  arr_hour
  dep_hour_minus2
  arr_hour_minus2
  origin_temp_f
  origin_dewpoint_f
  origin_humidity
  origin_feels_like_f
  origin_wind_kts
  origin_gust_kts
  origin_visibility
  origin_precip_in
  origin_wx_codes
  origin_is_rain
  origin_is_snow
  origin_is_fog
  origin_low_visibility
  origin_high_wind
  origin_severe_weather
  dest_temp_f
  dest_dewpoint_f
  dest_humidity
  dest_feels_like_f
  dest_wind_kts
  dest_gust_kts
  dest_visibility
  dest_precip_in
  dest_wx_codes
  dest_is_rain
  dest_is_snow
  dest_is_fog
  dest_low_visibility
  dest_high_wind
  dest_severe_weather
  is_weekend
  is_holiday
  origin_weather_missing
  dest_weather_missing
  carrier_delay_rate_30